## 1. Load and Analyze Dataset

In [18]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [19]:
# Load the composite dataset
df = pd.read_parquet('../data/composite_data.parquet')

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique tickers: {df['ticker'].nunique()}")

df.head()

Dataset shape: (39124, 23)
Date range: 2021-01-27 to 2021-08-15
Unique tickers: 300


,ticker,custom_sentiment,finbert_sentiment,date,on_volatile_date,adjacent_volatile_date,very_near_volatile_date,near_volatile_date,p-7,p-6,...,p-2,p-1,p+0,p+1,p+2,p+3,p+4,p+5,p+6,p+7
0,GME,-1.320963,-1.622274,2021-01-28,False,False,False,False,NaN,9.780000,...,36.994999,86.877502,48.400002,81.250000,56.25,22.500000,23.102501,13.375000,15.942500,15.000000
1,GME,-0.032856,0.572957,2021-01-28,False,False,False,False,NaN,9.780000,...,36.994999,86.877502,48.400002,81.250000,56.25,22.500000,23.102501,13.375000,15.942500,15.000000
2,GME,-0.462225,0.609483,2021-01-28,False,False,False,False,NaN,9.780000,...,36.994999,86.877502,48.400002,81.250000,56.25,22.500000,23.102501,13.375000,15.942500,15.000000
3,GME,-0.462225,0.514058,2021-01-28,False,False,False,False,NaN,9.780000,...,36.994999,86.877502,48.400002,81.250000,56.25,22.500000,23.102501,13.375000,15.942500,15.000000
4,AMC,-0.462225,0.514058,2021-01-28,False,False,False,False,NaN,29.700001,...,49.599998,199.000000,86.300003,132.600006,133.00,78.199997,89.699997,70.900002,68.300003,61.799999


In [20]:
# Analyze dataset structure
price_cols = [col for col in df.columns if col.startswith('p') and ('+' in col or '-' in col or col == 'p+0')]
volatile_cols = [col for col in df.columns if 'volatile' in col.lower()]

print(f"Price columns: {len(price_cols)}")
print(f"Volatile date indicators: {len(volatile_cols)}")
print(f"Most mentioned tickers:")
print(df['ticker'].value_counts().head(5))

df.sample(3)

Price columns: 15
Volatile date indicators: 4
Most mentioned tickers:
ticker
GME    12783
AMC     4545
DD      2699
NWS     1888
BB      1880
Name: count, dtype: int64


,ticker,custom_sentiment,finbert_sentiment,date,on_volatile_date,adjacent_volatile_date,very_near_volatile_date,near_volatile_date,p-7,p-6,...,p-2,p-1,p+0,p+1,p+2,p+3,p+4,p+5,p+6,p+7
979,BB,0.396513,0.203347,2021-01-28,False,False,False,False,NaN,12.790000,...,18.920000,25.100000,14.650000,14.10,14.630000,11.550000,12.000000,12.150000,13.2300,13.760000
33271,AMC,-0.891594,-1.239126,2021-04-28,False,False,False,False,96.599998,92.800003,...,115.000000,114.599998,108.500000,102.00,100.300003,97.099998,93.900002,91.699997,90.0000,95.099998
453,GME,-0.891594,0.193482,2021-01-28,False,False,False,False,NaN,9.780000,...,36.994999,86.877502,48.400002,81.25,56.250000,22.500000,23.102501,13.375000,15.9425,15.000000


## 2. Data Cleaning and Preprocessing

In [21]:
# Create a copy for processing
df_clean = df.copy()
print(f"Starting with {df_clean.shape[0]} rows")

# Ensure date column is datetime
if 'date' in df_clean.columns:
    df_clean['date'] = pd.to_datetime(df_clean['date'])

# Identify price columns
price_cols = [col for col in df_clean.columns if col.startswith('p') and ('+' in col or '-' in col or col == 'p+0')]
price_cols = sorted(price_cols, key=lambda x: int(x.replace('p', '').replace('+', '')))

print(f"Price columns identified: {len(price_cols)}")
print(f"Price columns: {price_cols}")

# Remove rows where there is no price data at all
valid_price_mask = df_clean[price_cols].notna().any(axis=1)
df_clean = df_clean[valid_price_mask].reset_index(drop=True)
print(f"After removing rows with no price data: {df_clean.shape[0]} rows")

# Check how much data we have for key price points
key_prices = ['p+0', 'p+1', 'p+2', 'p+3']  # Current and next few days for immediate 1 and 3 day predictions
for col in key_prices:
    if col in df_clean.columns:
        valid_pct = (df_clean[col].notna().sum() / len(df_clean)) * 100
        print(f"{col}: {valid_pct:.1f}% valid data")

# Remove duplicates if any
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Removed {initial_rows - len(df_clean)} duplicate rows")

print(f"\nFinal cleaned dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"Unique tickers: {df_clean['ticker'].nunique()}")

Starting with 39124 rows
Price columns identified: 15
Price columns: ['p-7', 'p-6', 'p-5', 'p-4', 'p-3', 'p-2', 'p-1', 'p+0', 'p+1', 'p+2', 'p+3', 'p+4', 'p+5', 'p+6', 'p+7']
After removing rows with no price data: 39124 rows
p+0: 100.0% valid data
p+1: 100.0% valid data
p+2: 100.0% valid data
p+3: 100.0% valid data
Removed 211 duplicate rows

Final cleaned dataset: 38913 rows × 23 columns
Date range: 2021-01-27 00:00:00 to 2021-08-15 00:00:00
Unique tickers: 300


## 3. Exploratory Data Analysis

In [22]:
# Calculate price returns for regression targets
if 'p+0' in df_clean.columns and 'p+1' in df_clean.columns:
    valid_mask = df_clean['p+0'].notna() & df_clean['p+1'].notna()
    df_clean.loc[valid_mask, 'return_1d'] = (df_clean.loc[valid_mask, 'p+1'] / df_clean.loc[valid_mask, 'p+0']) - 1

if 'p+0' in df_clean.columns and 'p+3' in df_clean.columns:
    valid_mask = df_clean['p+0'].notna() & df_clean['p+3'].notna()
    df_clean.loc[valid_mask, 'return_3d'] = (df_clean.loc[valid_mask, 'p+3'] / df_clean.loc[valid_mask, 'p+0']) - 1

if 'p+0' in df_clean.columns and 'p+7' in df_clean.columns:
    valid_mask = df_clean['p+0'].notna() & df_clean['p+7'].notna()
    df_clean.loc[valid_mask, 'return_7d'] = (df_clean.loc[valid_mask, 'p+7'] / df_clean.loc[valid_mask, 'p+0']) - 1

# Analyze return distributions
for col in ['return_1d', 'return_3d', 'return_7d']:
    if col in df_clean.columns and df_clean[col].notna().sum() > 0:
        returns = df_clean[col].dropna()
        print(f"\n{col}: {len(returns)} samples")
        print(f"  Mean: {returns.mean()*100:.2f}%, Std: {returns.std()*100:.2f}%")
        print(f"  Range: {returns.min()*100:.1f}% to {returns.max()*100:.1f}%")


return_1d: 38913 samples
  Mean: 8.82%, Std: 27.43%
  Range: -60.0% to 103.9%

return_3d: 38913 samples
  Mean: -11.03%, Std: 26.95%
  Range: -76.2% to 136.4%

return_7d: 32650 samples
  Mean: -13.42%, Std: 41.56%
  Range: -84.5% to 417.8%


## 4. Feature Engineering

In [23]:
# Create price-based features
# Price movement features
if 'p+0' in df_clean.columns and 'p-1' in df_clean.columns:
    df_clean['momentum_1d'] = (df_clean['p+0'] / df_clean['p-1']) - 1

if 'p+0' in df_clean.columns and 'p-3' in df_clean.columns:
    df_clean['momentum_3d'] = (df_clean['p+0'] / df_clean['p-3']) - 1

if 'p+0' in df_clean.columns and 'p-7' in df_clean.columns:
    df_clean['momentum_7d'] = (df_clean['p+0'] / df_clean['p-7']) - 1

# Price volatility
past_price_cols = [col for col in price_cols if '-' in col]
if len(past_price_cols) >= 3:
    past_prices = df_clean[past_price_cols].values
    df_clean['volatility'] = np.nanstd(past_prices, axis=1)

# Technical indicators
if 'p+0' in df_clean.columns:
    # Moving averages
    short_cols = ['p-2', 'p-1', 'p+0']
    long_cols = ['p-6', 'p-5', 'p-4', 'p-3', 'p-2', 'p-1', 'p+0']
    
    valid_short = df_clean[short_cols].notna().all(axis=1)
    valid_long = df_clean[long_cols].notna().all(axis=1)
    
    df_clean.loc[valid_short, 'sma_3d'] = df_clean.loc[valid_short, short_cols].mean(axis=1)
    df_clean.loc[valid_long, 'sma_7d'] = df_clean.loc[valid_long, long_cols].mean(axis=1)
    
    # MA crossover signal
    valid_ma = df_clean['sma_3d'].notna() & df_clean['sma_7d'].notna()
    df_clean.loc[valid_ma, 'ma_signal'] = (df_clean.loc[valid_ma, 'sma_3d'] > df_clean.loc[valid_ma, 'sma_7d']).astype(int)

# Time features
if 'date' in df_clean.columns:
    df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
    df_clean['is_monday'] = (df_clean['day_of_week'] == 0).astype(int)
    df_clean['is_friday'] = (df_clean['day_of_week'] == 4).astype(int)

# Volatile date features are already in the new data
volatile_cols = [col for col in df_clean.columns if 'volatile' in col.lower()]

print(f"Features created: momentum, volatility, moving averages, time indicators")
print(f"Dataset shape: {df_clean.shape}")

Features created: momentum, volatility, moving averages, time indicators
Dataset shape: (38913, 36)


## 5. Model Building and Training

In [24]:
# regression modeling
%pip install xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

# Select price-only features
feature_candidates = [
    'momentum_1d', 'momentum_3d', 'momentum_7d', 'volatility',
    'sma_3d', 'sma_7d', 'ma_signal', 'day_of_week',
    'is_monday', 'is_friday'
]

# Add volatile date indicators
volatile_cols = [col for col in df_clean.columns if 'volatile' in col.lower()]
feature_candidates.extend(volatile_cols)

# Filter to existing columns
available_features = [col for col in feature_candidates if col in df_clean.columns]
print(f"Price-only features: {len(available_features)}")

# Prepare dataset for 1-day return prediction
target_col = 'return_1d'
modeling_data = df_clean[available_features + [target_col, 'date', 'ticker']].copy()

# Remove missing targets
modeling_data = modeling_data.dropna(subset=[target_col])

# Remove incomplete rows
missing_threshold = len(available_features) * 0.5
feature_completeness = modeling_data[available_features].notna().sum(axis=1)
complete_data = modeling_data[feature_completeness >= missing_threshold].copy()

# Fill missing values
X = complete_data[available_features].fillna(complete_data[available_features].median())
y = complete_data[target_col]

print(f"Modeling dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target stats: mean={y.mean()*100:.2f}%, std={y.std()*100:.2f}%")

# Time-based split
split_idx = int(0.8 * len(complete_data))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

Note: you may need to restart the kernel to use updated packages.
Price-only features: 14
Modeling dataset: 38913 samples, 14 features
Target stats: mean=8.82%, std=27.43%
Train: 31130 samples, Test: 7783 samples



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\1042401\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# First, prepare datasets for multiple horizons
modeling_datasets = {}

for target_col in ['return_1d', 'return_3d', 'return_7d']:
    if target_col in df_clean.columns:
        horizon = target_col.replace('return_', '')
        modeling_data = df_clean[available_features + [target_col, 'date', 'ticker']].copy()
        modeling_data = modeling_data.dropna(subset=[target_col])
        
        feature_completeness = modeling_data[available_features].notna().sum(axis=1)
        complete_data = modeling_data[feature_completeness >= missing_threshold].copy()
        
        X = complete_data[available_features].fillna(complete_data[available_features].median())
        y = complete_data[target_col]
        
        split_idx = int(0.8 * len(complete_data))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        
        modeling_datasets[horizon] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test
        }

# Train models for all prediction horizons
all_results = {}
best_models = {}

for horizon in modeling_datasets.keys():
    print(f"\n Models for {horizon} predictions:")
    
    # Get data for this horizon
    X_train = modeling_datasets[horizon]['X_train']
    X_test = modeling_datasets[horizon]['X_test']
    y_train = modeling_datasets[horizon]['y_train']
    y_test = modeling_datasets[horizon]['y_test']
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    horizon_results = {}
    horizon_models = {}
    
    # 1. Baseline
    baseline_pred = np.full(len(y_test), y_train.mean())
    baseline_mae = mean_absolute_error(y_test, baseline_pred)
    baseline_r2 = 0.0
    horizon_results['Baseline'] = {'MAE': baseline_mae, 'R²': baseline_r2}
    horizon_models['Baseline'] = (None, None, baseline_pred)
    
    # 2. Linear Regression
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    lr_pred = lr_model.predict(X_test_scaled)
    lr_mae = mean_absolute_error(y_test, lr_pred)
    lr_r2 = r2_score(y_test, lr_pred)
    horizon_results['Linear'] = {'MAE': lr_mae, 'R²': lr_r2}
    horizon_models['Linear'] = (lr_model, scaler, lr_pred)
    
    # 3. Ridge Regression
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_pred = ridge_model.predict(X_test_scaled)
    ridge_mae = mean_absolute_error(y_test, ridge_pred)
    ridge_r2 = r2_score(y_test, ridge_pred)
    horizon_results['Ridge'] = {'MAE': ridge_mae, 'R²': ridge_r2}
    horizon_models['Ridge'] = (ridge_model, scaler, ridge_pred)
    
    # 4. Random Forest
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_r2 = r2_score(y_test, rf_pred)
    horizon_results['Random Forest'] = {'MAE': rf_mae, 'R²': rf_r2}
    horizon_models['Random Forest'] = (rf_model, None, rf_pred)
    
    # Find best model for this horizon
    best_model_name = max(horizon_results.keys(), key=lambda k: horizon_results[k]['R²'])
    
    all_results[horizon] = horizon_results
    best_models[horizon] = {
        'name': best_model_name,
        'model': horizon_models[best_model_name],
        'metrics': horizon_results[best_model_name],
        'y_test': y_test
    }
    
    print(f"  Best: {best_model_name} (R²={horizon_results[best_model_name]['R²']:.4f}, MAE={horizon_results[best_model_name]['MAE']*100:.3f}%)")

# Summary comparison across
print("\nPerformance Summary Across Horizons:")
for horizon in all_results.keys():
    best_name = best_models[horizon]['name']
    best_r2 = best_models[horizon]['metrics']['R²']
    best_mae = best_models[horizon]['metrics']['MAE']
    print(f"{horizon}: {best_name} - R²={best_r2:.4f}, MAE={best_mae*100:.3f}%")


 Models for 1d predictions:
  Best: Random Forest (R²=0.2101, MAE=2.964%)

 Models for 3d predictions:
  Best: Random Forest (R²=0.2101, MAE=2.964%)

 Models for 3d predictions:
  Best: Baseline (R²=0.0000, MAE=15.739%)

 Models for 7d predictions:
  Best: Baseline (R²=0.0000, MAE=15.739%)

 Models for 7d predictions:
  Best: Baseline (R²=0.0000, MAE=18.164%)

Performance Summary Across Horizons:
1d: Random Forest - R²=0.2101, MAE=2.964%
3d: Baseline - R²=0.0000, MAE=15.739%
7d: Baseline - R²=0.0000, MAE=18.164%
  Best: Baseline (R²=0.0000, MAE=18.164%)

Performance Summary Across Horizons:
1d: Random Forest - R²=0.2101, MAE=2.964%
3d: Baseline - R²=0.0000, MAE=15.739%
7d: Baseline - R²=0.0000, MAE=18.164%


## 6. Multi-Day Model Evaluation and Analysis

In [ ]:
print("Analysis by Prediction Horizon (number of days past t)")

for horizon in best_models.keys():
    print(f"\n{horizon.upper()} PREDICTIONS:")
    
    model_name = best_models[horizon]['name']
    model_tuple = best_models[horizon]['model']
    metrics = best_models[horizon]['metrics']
    y_test = best_models[horizon]['y_test']
    
    model_obj, scaler, predictions = model_tuple
    
    print(f"  Best model: {model_name}")
    print(f"  R²: {metrics['R²']:.4f}")
    print(f"  MAE: {metrics['MAE']*100:.3f}%")
    
    # Feature importance analysis
    if hasattr(model_obj, 'feature_importances_'):
        # Random Forest
        feature_importance = pd.DataFrame({
            'feature': available_features,
            'importance': model_obj.feature_importances_
        }).sort_values('importance', ascending=False)
        print(f"  Top 3 features:")
        for i, (_, row) in enumerate(feature_importance.head(3).iterrows()):
            print(f"    {i+1}. {row['feature']}: {row['importance']:.4f}")
    elif hasattr(model_obj, 'coef_'):
        # Linear/Ridge
        feature_importance = pd.DataFrame({
            'feature': available_features,
            'coefficient': np.abs(model_obj.coef_)
        }).sort_values('coefficient', ascending=False)
        print(f"  Top 3 features:")
        for i, (_, row) in enumerate(feature_importance.head(3).iterrows()):
            print(f"    {i+1}. {row['feature']}: {row['coefficient']:.4f}")
    
    # Prediction quality analysis
    if metrics['R²'] > 0:
        correlation = np.corrcoef(y_test, predictions)[0,1]
        print(f"  Prediction correlation: {correlation:.4f}")

Analysis by Days Past

1D PREDICTIONS:
  Best model: Random Forest
  R²: 0.2101
  MAE: 2.964%
  Top 3 features:
    1. momentum_1d: 0.7693
    2. momentum_7d: 0.0793
    3. momentum_3d: 0.0580
  Prediction correlation: 0.4906

3D PREDICTIONS:
  Best model: Baseline
  R²: 0.0000
  MAE: 15.739%

7D PREDICTIONS:
  Best model: Baseline
  R²: 0.0000
  MAE: 18.164%


In [ ]:
print("\nMulti-Day Analysis and Conclusions:")
print("Prediction difficulty increases with horizon (number of days past t):")

for horizon in ['1d', '3d', '7d']:
    if horizon in best_models:
        r2 = best_models[horizon]['metrics']['R²']
        mae = best_models[horizon]['metrics']['MAE']
        model_name = best_models[horizon]['name']
        print(f"{horizon}: {model_name} - R²={r2:.4f}, MAE={mae*100:.3f}%")

# Find best overall horizon
best_overall_horizon = max(best_models.keys(), key=lambda h: best_models[h]['metrics']['R²'])
best_r2 = best_models[best_overall_horizon]['metrics']['R²']

print(f"\nBest predictive horizon: {best_overall_horizon} (R²={best_r2:.4f})")

if best_overall_horizon == '1d':
    print("Short-term price movements are most predictable")
elif best_overall_horizon == '7d':
    print("Longer-term trends are more predictable than daily noise")
else:
    print("Medium-term (3-day) predictions work best")



Multi-Day Analysis and Conclusions:
Prediction difficulty increases with horizon (number of days past t):
1d: Random Forest - R²=0.2101, MAE=2.964%
3d: Baseline - R²=0.0000, MAE=15.739%
7d: Baseline - R²=0.0000, MAE=18.164%

Best predictive horizon: 1d (R²=0.2101)
Short-term price movements are most predictable
